# YOLO26 Inference - VinBigData test set -> submission.csv
Logic va dinh dang submission theo notebook tham khao `vinbigdata-cxr-ad-yolov5-14-class-infer` (chi doi phan chay model sang YOLO26 / Ultralytics API).

## 1. Setup

In [9]:
!pip install -q ultralytics kaggle
from ultralytics import YOLO
import numpy as np, pandas as pd
import os, glob, shutil
from tqdm.notebook import tqdm

# Mount Google Drive (noi luu checkpoint)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/Thực tập/VinBigData Chest X-ray Abnormalities Detection/YOLO26 distill/train vinbigdata"
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)


ERROR: Operation cancelled by user
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. CONFIG

In [10]:
DATA_DIR = "/content/vbd_data"

# Cau truc that cua dataset (xac nhan tu notebook tham khao):
# vinbigdata-1024-image-dataset giai nen ra co thu muc goc "vinbigdata/" chua test/, test.csv, train/, ...
TEST_DIR = f"{DATA_DIR}/vinbigdata/test"
TEST_CSV = f"{DATA_DIR}/vinbigdata/test.csv"

# DRIVE_PROJECT_DIR = "/content/drive/MyDrive/VinBigData_YOLO26"
RUN_NAME = "yolo26n_vbd"
WEIGHTS_PATH = f"{DRIVE_PROJECT_DIR}/runs/{RUN_NAME}/weights/best.pt"

IMG_SIZE = 640
CONF_THRES = 0.01
IOU_THRES = 0.4


## 3. Tai anh test tu Kaggle (neu chua co)

In [11]:
if not os.path.exists('/root/.kaggle/kaggle.json'):
    from google.colab import files
    print("Upload kaggle.json:")
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    shutil.move(list(uploaded.keys())[0], '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.exists(TEST_DIR):
    !kaggle datasets download -d awsaf49/vinbigdata-1024-image-dataset -p {DATA_DIR} --unzip


In [12]:
test_df = pd.read_csv(TEST_CSV)
test_df.head()


,image_id,width,height
0,83caa8a85e03606cf57e49147d7ac569,2304,2880
1,7550347fa2bb96c2354a3716dfa3a69c,2538,3095
2,74b23792db329cff5843e36efb8aa65a,2788,3120
3,94568a546be103177cb582d3e91cd2d8,1994,2430
4,6da36354fc904b63bc03eb3884e0c35c,2056,2376


## 4. Load model da train

In [13]:
model = YOLO(WEIGHTS_PATH)


## 5. Inference tren tap test
Cung logic voc-format nhu notebook tham khao: box yolo (normalized xywh) -> voc (x1 y1 x2 y2) theo width/height GOC cua anh (lay tu test.csv), khong phai kich thuoc anh dua vao model.

In [14]:
def yolo2voc(image_height, image_width, bboxes):
    """
    yolo => [xmid, ymid, w, h] (normalized)
    voc  => [x1, y1, x2, y2]
    """
    bboxes = bboxes.copy().astype(float)
    bboxes[..., [0, 2]] = bboxes[..., [0, 2]] * image_width
    bboxes[..., [1, 3]] = bboxes[..., [1, 3]] * image_height
    bboxes[..., [0, 1]] = bboxes[..., [0, 1]] - bboxes[..., [2, 3]] / 2
    bboxes[..., [2, 3]] = bboxes[..., [0, 1]] + bboxes[..., [2, 3]]
    return bboxes


In [15]:
image_ids = []
PredictionStrings = []

results = model.predict(
    source=TEST_DIR,
    conf=CONF_THRES,
    iou=IOU_THRES,
    imgsz=IMG_SIZE,
    device=0,
    stream=True,
    verbose=False,
)

for r in tqdm(results, total=len(test_df)):
    image_id = os.path.splitext(os.path.basename(r.path))[0]
    w, h = test_df.loc[test_df.image_id == image_id, ["width", "height"]].values[0]

    if len(r.boxes) == 0:
        image_ids.append(image_id)
        PredictionStrings.append("14 1 0 0 1 1")
        continue

    cls = r.boxes.cls.cpu().numpy().reshape(-1, 1)
    conf = r.boxes.conf.cpu().numpy().reshape(-1, 1)
    xywhn = r.boxes.xywhn.cpu().numpy()

    data = np.concatenate([cls, conf, xywhn], axis=1)  # [class, conf, x, y, w, h]
    voc = np.round(yolo2voc(h, w, data[:, 2:]))
    rounded = np.round(np.concatenate([data[:, :2], voc], axis=1), 3)

    bboxes = list(rounded.reshape(-1).astype(str))
    for idx in range(len(bboxes)):
        bboxes[idx] = str(int(float(bboxes[idx]))) if idx % 6 != 1 else bboxes[idx]

    image_ids.append(image_id)
    PredictionStrings.append(" ".join(bboxes))


  0%|          | 0/3000 [00:00<?, ?it/s]

## 6. Xuat submission.csv

In [16]:
pred_df = pd.DataFrame({"image_id": image_ids, "PredictionString": PredictionStrings})
sub_df = pd.merge(test_df, pred_df, on="image_id", how="left").fillna("14 1 0 0 1 1")
sub_df = sub_df[["image_id", "PredictionString"]]

sub_df.to_csv("/content/submission.csv", index=False)
shutil.copy("/content/submission.csv", f"{DRIVE_PROJECT_DIR}/submission.csv")

sub_df.tail()


,image_id,PredictionString
2995,7f5503caa936a623b4388fbd88e890c5,3 0.17 996 1640 2095 2013 3 0.105 994 1648 211...
2996,c97e54a78bab9c05ce2e04fe6c284bcd,0 0.652 1704 960 2079 1464 3 0.381 1370 1719 2...
2997,33218cf183c1224a74ccfb514e827e15,3 0.284 867 1324 1886 1781 3 0.243 928 1372 18...
2998,04b700c4815f088728db9f093c739707,0 0.103 1326 1047 1602 1357 11 0.057 683 584 1...
2999,14da9051525bd2504dd56938f92644ef,0 0.269 997 813 1202 1033 3 0.266 754 1373 150...
